# Imports e Bibliotecas

In [1]:
import pandas as pd
import numpy as np
from enum import Enum

# Dataset

In [3]:
deep_seek_v3_coleta = pd.read_csv("/content/Corpus_DeepSeek-V3.csv")
gpt_2_coleta = pd.read_csv("/content/Corpus_gpt-2.csv")
grok_1_coleta = pd.read_csv("/content/Corpus_grok-1.csv")
llama_coleta = pd.read_csv("/content/Corpus_llama.csv")
llama3_coleta = pd.read_csv("/content/Corpus_llama3.csv")

In [4]:
deep_seek_v3_coleta['Repositorio'] = 'DeepSeek_V3'
gpt_2_coleta['Repositorio'] = 'GPT_2'
grok_1_coleta['Repositorio'] = 'Grok_1'
llama_coleta['Repositorio'] = 'Llama'
llama3_coleta['Repositorio'] = 'Llama3'

In [5]:
todos = pd.concat([deep_seek_v3_coleta, gpt_2_coleta, grok_1_coleta, llama_coleta, llama3_coleta], ignore_index=True)

## Adicionando coluna $Validadas$
Representa se a *issue* está nos conjuntos das *issues* validadas como relacionadas à ética ou não

In [6]:
id_issues_validadas = pd.read_csv("/content/validadas.csv")

In [7]:
def addColunaValidada(dataset, id_issues_validadas):
  dataset['Validada'] = dataset['IdIssue'].isin(id_issues_validadas['IdIssue'])
  return dataset

In [8]:
todos = addColunaValidada(todos,id_issues_validadas)

## Identificando quem contribuiu para discussão

In [9]:
contribuintes = pd.read_csv('/content/ValidacaoExtra.csv')

In [10]:
contribuintes['Soma_Total'] = contribuintes['QtdComentarios'] + contribuintes['Info']
soma_agrupada = contribuintes.groupby(['IdIssue', 'Autor'])['Soma_Total'].sum().reset_index()
todos = todos.merge(
    soma_agrupada,
    left_on=['IdIssue', 'AutorComentario'],
    right_on=['IdIssue', 'Autor'],
    how='left'
)
todos.rename(columns={'Soma_Total': 'Volume'}, inplace=True)
todos['Volume'] = todos['Volume'].fillna(0)
todos.drop(columns=['Autor'], inplace=True)

## Anonimização

In [11]:
def anonimizaAutores(dataset):
  cp = dataset.copy()
  cp['AutorComentario'] = pd.factorize(cp['AutorComentario'])[0]
  cp['IdIssue'] = pd.factorize(cp['IdIssue'])[0]
  cp = cp.drop(columns=['LinkIssue','NumeroComentario','Comentario','Info'])
  return cp

### Split Dataset
1. Dataset somente com *issues* validadas (observa quem contribuiu ou não)
3. Dataset somente com *issues* invalidadas
---
> Permite avaliar se existe transversalidade fora do contexto de investigação de aspectos éticos (é mais um plus)

> Quero avaliar também a visão geral, ou seja, se um usuário que abordou aspecto ético também abordou outros temas não relacionados a aspectos éticos ao longo dos repositórios

> Vou analisar também o fato de ter participado numa discussão validada, se há a transversalidade

In [12]:
todos_anon=anonimizaAutores(todos)
todos_validadas=todos_anon[todos_anon['Validada']]
todos_invalidadas=todos_anon[todos_anon['Validada']==False]

# Estrutura de Dados

In [13]:
ids_validados = set(todos_validadas['IdIssue'].tolist())

In [14]:
ids_invalidados = set(todos_invalidadas['IdIssue'].tolist())

In [15]:
class Repositorios(Enum):
  DeepSeek_V3 = 1
  Grok_1 = 2
  GPT_2 = 3
  Llama = 4
  Llama3 = 5

In [ ]:
#vou usar as validadas, a visualização pode colorir cada usuario com uma cor e o tamanho do no a qtd de comentarios feitos
#interessante montar rede por aspecto abordado ... ?
#o objetivo é ver a transversalidade do discurso

In [19]:
class Issue:
  def __init__(self, id_issue:int, repositorio:Repositorios):
    self.id_issue = id_issue
    self.repositorio = repositorio
    self.relacionada_etica = self.verificaValidada()

  def verificaValidada(self):
    if self.id_issue in ids_validados:
      return True
    else:
      return False

In [109]:
class No_Usuario:
  def __init__(self, id_participante:int, aspecto_etico_abordado:bool, issues:list[Issue]): #devo tirar aspecto_etico, mas ainda n tenho ctz
    self.id_participante = id_participante
    self.aspecto_etico_abordado = aspecto_etico_abordado #foi o usuário quem abordou o aspecto - sim ou não
    self.issues = issues
    self.repositorio_predominante = Repositorios(self.predominancia_repos()).name #indica o repo que um usuário mais participou
    self.volume_discussao_aspectos_eticos = self.discussao_aspectos_eticos()

  def predominancia_repos(self):
    contabiliza_repo = {Repositorios(1).name:0,
                        Repositorios(2).name:0,
                        Repositorios(3).name:0,
                        Repositorios(4).name:0,
                        Repositorios(5).name:0}

    for i in self.issues:
      if i.repositorio.name in contabiliza_repo.keys():
        contabiliza_repo[i.repositorio.name]+=1

    maior = 0
    for i in contabiliza_repo.values():
      if maior < i:
        maior = i

    return maior

  def discussao_aspectos_eticos(self):
    #1.verificar se usuario esta em issue validada -> self.issues.id_issue in ids_validadas
    #2.nas issues validadas do usuario, buscar em todas_validadas o id_participante dele em AutorComentario
    #3.agrupar usuarios filtrados por AutorComentario e somar dado na coluna Volume em todas_validadas pelos agrupamentos
    #4.retornar resultado

    ids_issues_usuario = [i.id_issue for i in self.issues]

    df_filtrado = todos_validadas[todos_validadas['IdIssue'].isin(ids_issues_usuario)]

    filtro_autor = df_filtrado['AutorComentario'] == self.id_participante
    df_filtrado = df_filtrado[filtro_autor]

    volume_total = int(df_filtrado['Volume'].sum())

    return volume_total


In [ ]:
#matriz de adjacência, nós se conectam pela quantidade de issues em ambos. Se 2 usuarios participaram de 3 issues ao mesmo tempo, eles tem 3 conexoes
# a aresta pode ser rotulada com id do repo, visualmente pode ser uma cor
class Grafo:
  def __init__(self):
    pass
  def inserir_no(self):
    pass
  def criar_aresta(self):
    pass
  def vizinhos(self):
    pass
  def dfs(self):
    pass
  def brandes(self):
    pass

# Análise